# Exploração: LPCON - Nurban 901

Objetivo: entender a estrutura do arquivo e definir como extrair dados para o pipeline blu.

In [1]:
from pathlib import Path

import openpyxl
import pandas as pd

FILE = Path('LPCON - Nurban 901 - REV.01.xlsx')
wb = openpyxl.load_workbook(FILE, data_only=True)  # data_only=True: resolve fórmulas para o valor cacheado
print('Sheets:', wb.sheetnames)

Sheets: ['familias', 'aditivos', 'lançamentos', 'orçamento', 'Aux', 'resumo']


/opt/homebrew/lib/python3.14/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


## 1. Mapa de sheets

Visão rápida de quantas linhas/colunas e as primeiras linhas de cada sheet.

In [2]:
for name in wb.sheetnames:
    ws = wb[name]
    rows = list(ws.iter_rows(min_row=1, max_row=4, values_only=True))
    print(f"\n{'='*60}")
    print(f"Sheet: '{name}'  ({ws.max_row} linhas x {ws.max_column} colunas)")
    for i, row in enumerate(rows, 1):
        non_none = [v for v in row if v is not None]
        print(f"  row{i}: {non_none[:8]}")


Sheet: 'familias'  (10 linhas x 2 colunas)
  row1: []
  row2: ['familias']
  row3: ['MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO']
  row4: ['DEMOLIÇÕES']

Sheet: 'aditivos'  (272 linhas x 15 colunas)
  row1: []
  row2: ['MATERIAL EXTRA', 0]
  row3: ['DATA DE RECEBIMENTO', 'QUINZENA', 'FORNECEDOR', 'QTD', 'UNID.', 'DESCRIÇÃO', 'MARCA', 'CENTRO DE CUSTO']
  row4: []

Sheet: 'lançamentos'  (212 linhas x 12 colunas)
  row1: []
  row2: ['MATERIAL', 'INSUMOS', 'VENDA']
  row3: ['MÃO DE OBRA', '#', 'DATA CUSTO', 'FORNECEDOR', 'MAT ou MDO', 'CENTRO DE CUSTO - FAMILIA', 'CENTRO DE CUSTO - SUB FAMILIA', 'NÚMERO   NF']
  row4: [1, datetime.datetime(2025, 9, 12, 0, 0), 'BAZAR TODA OBRA', 'MAT', 'OBRA CIVIL', 'ELEVAÇÃO DE ALVENARIA', 21.9]

Sheet: 'orçamento'  (65 linhas x 19 colunas)
  row1: []
  row2: ['ORÇAMENTO NURBAN']
  row3: ['ITEM', 'DESCRIÇÃO', 'QUANTIDADE', 'UNIDADE', '% ABC', 'UNITÁRIO MATERIAL', 'TOTAL MATERIAL', 'UNITÁRIO MDO']
  row4: [1.15, 1.28, 0.1533, 1.15, 1.28, 0.1533]

Sheet: 'Aux'  

## 2. Sheet principal: `lançamentos`

Contém os lançamentos de custo (MAT e MDO) do projeto.

**Estrutura observada:**
- Row 1: vazia
- Row 2: labels de seção — `MATERIAL`, `INSUMOS`, `VENDA` (não são headers de coluna)
- Row 3: headers reais de coluna
- Row 4+: dados

In [3]:
ws_lanc = wb['lançamentos']

# Mostrar as primeiras linhas brutas para entender a estrutura
print('Linhas brutas (1-5):')
for i, row in enumerate(ws_lanc.iter_rows(min_row=1, max_row=5, values_only=True), 1):
    print(f'  row{i}: {list(row)}')

Linhas brutas (1-5):
  row1: [None, None, None, None, None, None, None, None, None, None, None, None]
  row2: ['MATERIAL', 'INSUMOS', None, None, None, None, None, None, None, None, 'VENDA', None]
  row3: ['MÃO DE OBRA', '#', 'DATA CUSTO', 'FORNECEDOR', 'MAT ou MDO', 'CENTRO DE CUSTO - FAMILIA', 'CENTRO DE CUSTO - SUB FAMILIA', 'NÚMERO   NF', 'VALOR', None, None, None]
  row4: [None, 1, datetime.datetime(2025, 9, 12, 0, 0), 'BAZAR TODA OBRA', 'MAT', 'OBRA CIVIL', 'ELEVAÇÃO DE ALVENARIA', None, 21.9, None, None, None]
  row5: [None, 2, datetime.datetime(2025, 9, 12, 0, 0), 'A ILUMINADA', 'MAT', 'INSTALAÇÕES', 'INSTALAÇÃO ELÉTRICA ', None, 3216.39, None, None, None]


In [4]:
# Extrair com header na row 3 (index 2), dados a partir de row 4
all_rows = list(ws_lanc.iter_rows(values_only=True))

header_row_idx = 2  # 0-indexed → row 3
headers = [str(v).strip() if v is not None else f'col_{i}' for i, v in enumerate(all_rows[header_row_idx])]
print('Headers encontrados:', headers)

data_rows = all_rows[header_row_idx + 1:]
df_lanc = pd.DataFrame(data_rows, columns=headers)

# Remover linhas completamente vazias
df_lanc = df_lanc.dropna(how='all')
print(f'\nShape após remover linhas vazias: {df_lanc.shape}')
df_lanc.head(10)

Headers encontrados: ['MÃO DE OBRA', '#', 'DATA CUSTO', 'FORNECEDOR', 'MAT ou MDO', 'CENTRO DE CUSTO - FAMILIA', 'CENTRO DE CUSTO - SUB FAMILIA', 'NÚMERO   NF', 'VALOR', 'col_9', 'col_10', 'col_11']

Shape após remover linhas vazias: (199, 12)


,MÃO DE OBRA,#,DATA CUSTO,FORNECEDOR,MAT ou MDO,CENTRO DE CUSTO - FAMILIA,CENTRO DE CUSTO - SUB FAMILIA,NÚMERO NF,VALOR,col_9,col_10,col_11
0,None,1.0,2025-09-12,BAZAR TODA OBRA,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA,NaN,21.90,None,None,NaN
1,None,2.0,2025-09-12,A ILUMINADA,MAT,INSTALAÇÕES,INSTALAÇÃO ELÉTRICA,NaN,3216.39,None,None,NaN
2,None,3.0,2025-09-12,EDSON SANTOS MATERIAL,MAT,INSTALAÇÕES,INSTALAÇÃO HIDRÁULICA,NaN,30.00,None,None,NaN
3,None,4.0,2025-09-12,EDSON SANTOS MATERIAL,MAT,INSTALAÇÕES,INSTALAÇÃO HIDRÁULICA,NaN,38.75,None,None,NaN
4,None,5.0,2025-09-12,ÁGUAS CLARA,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,OUTROS,NaN,36.00,None,None,NaN
5,None,6.0,2025-09-12,ZAMACON,MAT,DEMOLIÇÕES,DEMOLIÇÃO DE ALVENARIA,33745.0,240.00,None,None,NaN
6,None,7.0,2025-09-12,ZAMACON,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA,33745.0,1058.50,None,None,NaN
7,None,8.0,2025-09-12,AL GOSTO,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,OUTROS,NaN,60.00,None,None,NaN
8,None,9.0,2025-09-12,ZAMACON,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,"EQUIPAMENTOS, EPIS E EPCS",33745.0,916.80,None,None,NaN
9,None,10.0,2025-09-15,ZAMACON,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA,33796.0,1049.00,None,None,NaN


In [5]:
# Tipos de dados
df_lanc.dtypes

MÃO DE OBRA                              object
#                                       float64
DATA CUSTO                       datetime64[us]
FORNECEDOR                                  str
MAT ou MDO                                  str
CENTRO DE CUSTO - FAMILIA                   str
CENTRO DE CUSTO - SUB FAMILIA               str
NÚMERO   NF                             float64
VALOR                                   float64
col_9                                    object
col_10                                   object
col_11                                  float64
dtype: object

In [6]:
# Estatísticas básicas das colunas numéricas
df_lanc.describe(include='all')

,MÃO DE OBRA,#,DATA CUSTO,FORNECEDOR,MAT ou MDO,CENTRO DE CUSTO - FAMILIA,CENTRO DE CUSTO - SUB FAMILIA,NÚMERO NF,VALOR,col_9,col_10,col_11
count,0,199.000000,199,199,199,199,192,31.000000,199.000000,0,0,2.000000
unique,0,NaN,NaN,46,2,7,17,NaN,NaN,0,0,NaN
top,NaN,NaN,NaN,ZAMACON,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,ELEVAÇÃO DE ALVENARIA,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,33,161,69,56,NaN,NaN,NaN,NaN,NaN
mean,NaN,100.000000,2025-12-16 01:55:46.733668,NaN,NaN,NaN,NaN,38152.096774,507.514121,NaN,NaN,42500.000000
min,NaN,1.000000,2025-09-12 00:00:00,NaN,NaN,NaN,NaN,11279.000000,1.700000,NaN,NaN,20000.000000
25%,NaN,50.500000,2025-10-10 00:00:00,NaN,NaN,NaN,NaN,33796.000000,34.000000,NaN,NaN,31250.000000
50%,NaN,100.000000,2025-12-08 00:00:00,NaN,NaN,NaN,NaN,34026.000000,160.000000,NaN,NaN,42500.000000
75%,NaN,149.500000,2026-02-11 12:00:00,NaN,NaN,NaN,NaN,34554.500000,656.600000,NaN,NaN,53750.000000
max,NaN,199.000000,2026-04-24 00:00:00,NaN,NaN,NaN,NaN,229631.000000,3400.000000,NaN,NaN,65000.000000


## 3. Coluna `MAT ou MDO` — tipo de lançamento

Discrimina se o lançamento é material (MAT) ou mão de obra (MDO).

In [7]:
tipo_col = 'MAT ou MDO'
print(df_lanc[tipo_col].value_counts(dropna=False))

MAT ou MDO
MAT    161
MDO     38
Name: count, dtype: int64


## 4. Centros de custo

`CENTRO DE CUSTO - FAMILIA` e `CENTRO DE CUSTO - SUB FAMILIA` formam a hierarquia de categorias.

In [8]:
familia_col = 'CENTRO DE CUSTO - FAMILIA'
sub_col = 'CENTRO DE CUSTO - SUB FAMILIA'

hierarquia = df_lanc.groupby([familia_col, sub_col])['VALOR'].agg(['count', 'sum']).reset_index()
hierarquia.columns = ['Família', 'Sub-família', 'Qtd lançamentos', 'Total R$']
hierarquia['Total R$'] = hierarquia['Total R$'].map('R$ {:,.2f}'.format)
hierarquia.sort_values('Família')

,Família,Sub-família,Qtd lançamentos,Total R$
0,DEMOLIÇÕES,ABERTURA DE RASGOS DIVERSOS EM ALVENARIA E CON...,1,R$ 285.00
1,DEMOLIÇÕES,BOTA FORA EM CAÇAMBAS LEGAIS,4,"R$ 1,700.00"
2,DEMOLIÇÕES,DEMOLIÇÃO DE ALVENARIA,7,"R$ 7,500.00"
3,ESTRUTURA E DE CONCRETO ARMADO,EXECUÇÃO DE BANCADA EM CONCRETO,1,R$ 187.60
4,ESTRUTURA E DE CONCRETO ARMADO,EXECUÇÃO DE ESCADA DE CONCRETO ARMADO,8,"R$ 3,358.61"
5,INSTALAÇÕES,INSTALAÇÃO DE AR CONDICIONADO,5,"R$ 3,604.30"
6,INSTALAÇÕES,INSTALAÇÃO ELÉTRICA,20,"R$ 17,012.59"
7,INSTALAÇÕES,INSTALAÇÃO HIDRÁULICA,11,"R$ 2,171.05"
10,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,OUTROS,50,"R$ 1,411.85"
8,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,ACOMPANHAMENTO TÉCNICO,9,"R$ 2,869.40"


## 5. Sheet `aditivos` — Materiais extras

Lançamentos adicionais não incluídos no orçamento original.

In [9]:
ws_adit = wb['aditivos']

print('Linhas brutas (1-5):')
for i, row in enumerate(ws_adit.iter_rows(min_row=1, max_row=5, values_only=True), 1):
    print(f'  row{i}: {[v for v in row if v is not None]}')

Linhas brutas (1-5):
  row1: []
  row2: ['MATERIAL EXTRA', 0]
  row3: ['DATA DE RECEBIMENTO', 'QUINZENA', 'FORNECEDOR', 'QTD', 'UNID.', 'DESCRIÇÃO', 'MARCA', 'CENTRO DE CUSTO', 'NÚMERO   NF', 'VALOR']
  row4: []
  row5: []


In [10]:
# Header em row 3 (idx 2), dados a partir de row 4
adit_rows = list(ws_adit.iter_rows(values_only=True))
adit_headers = [str(v).strip() if v is not None else f'col_{i}' for i, v in enumerate(adit_rows[2])]
print('Headers aditivos:', adit_headers)

df_adit = pd.DataFrame(adit_rows[3:], columns=adit_headers)
df_adit = df_adit.dropna(how='all')

# Filtrar apenas linhas com VALOR preenchido
valor_col = next((c for c in adit_headers if 'VALOR' in str(c).upper()), None)
if valor_col:
    df_adit = df_adit[df_adit[valor_col].notna()]

print(f'Shape: {df_adit.shape}')
df_adit.head()

Headers aditivos: ['col_0', 'DATA DE RECEBIMENTO', 'QUINZENA', 'FORNECEDOR', 'QTD', 'UNID.', 'DESCRIÇÃO', 'MARCA', 'CENTRO DE CUSTO', 'NÚMERO   NF', 'VALOR', 'col_11', 'col_12', 'col_13', 'col_14']
Shape: (0, 15)


,col_0,DATA DE RECEBIMENTO,QUINZENA,FORNECEDOR,QTD,UNID.,DESCRIÇÃO,MARCA,CENTRO DE CUSTO,NÚMERO NF,VALOR,col_11,col_12,col_13,col_14


## 6. Sheet `orçamento` — Planejado vs real

Contém o orçamento original com valores unitários, MDO e BDI.

In [11]:
ws_orc = wb['orçamento']

print('Linhas brutas (1-6):')
for i, row in enumerate(ws_orc.iter_rows(min_row=1, max_row=6, values_only=True), 1):
    non_none = [v for v in row if v is not None]
    print(f'  row{i}: {non_none[:10]}')

Linhas brutas (1-6):
  row1: []
  row2: ['ORÇAMENTO NURBAN']
  row3: ['ITEM', 'DESCRIÇÃO', 'QUANTIDADE', 'UNIDADE', '% ABC', 'UNITÁRIO MATERIAL', 'TOTAL MATERIAL', 'UNITÁRIO MDO', 'TOTAL MDO', 'SUBTOTAL']
  row4: [1.15, 1.28, 0.1533, 1.15, 1.28, 0.1533]
  row5: [1, '1. MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO', 0.19076250607690612, 11000, 36576, 47576, 59467.28]
  row6: ['1.1 ACOMPANHAMENTO TÉCNICO', 6, 'MÊS', 0.14665649533943415, 0, 0, 6096, 36576, 36576, 46817.28]


In [12]:
# Header em row 3 (idx 2), dados a partir de row 5 (row 4 tem parâmetros BDI)
orc_rows = list(ws_orc.iter_rows(values_only=True))
orc_headers = [str(v).strip() if v is not None else f'col_{i}' for i, v in enumerate(orc_rows[2])]
print('Headers orçamento:', orc_headers)

df_orc = pd.DataFrame(orc_rows[4:], columns=orc_headers)
df_orc = df_orc.dropna(how='all')
df_orc.head(10)

Headers orçamento: ['col_0', 'ITEM', 'DESCRIÇÃO', 'QUANTIDADE', 'UNIDADE', '% ABC', 'UNITÁRIO MATERIAL', 'TOTAL MATERIAL', 'UNITÁRIO MDO', 'TOTAL MDO', 'SUBTOTAL', 'TOTAL C/ BDI', 'BDI MAT', 'BDI MDO', 'NF', 'col_15', 'BDI MAT', 'BDI MDO', 'NF']


,col_0,ITEM,DESCRIÇÃO,QUANTIDADE,UNIDADE,% ABC,UNITÁRIO MATERIAL,TOTAL MATERIAL,UNITÁRIO MDO,TOTAL MDO,SUBTOTAL,TOTAL C/ BDI,BDI MAT,BDI MDO,NF,col_15,BDI MAT,BDI MDO,NF
0,None,1.0,1. MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,NaN,NaN,0.190763,NaN,11000.0000,NaN,36576,47576,59467.28,None,None,None,None,NaN,NaN,None
1,None,NaN,1.1 ACOMPANHAMENTO TÉCNICO,6.00000,MÊS,0.146656,0.0,0.0000,6096.0,36576,36576,46817.28,None,None,None,None,NaN,NaN,None
2,None,NaN,"1.2 EQUIPAMENTOS, EPIS E EPCS",6.00000,MÊS,0.036087,1500.0,9000.0000,0.0,0,9000,10350,None,None,None,None,NaN,NaN,None
3,None,NaN,1.3 OUTROS,1.00000,VB,0.008019,2000.0,2000.0000,0.0,0,2000,2300,None,None,None,None,NaN,NaN,None
4,None,2.0,2. DEMOLIÇÕES,NaN,NaN,0.092237,NaN,8583.6175,NaN,14420.2525,23003.87,28329.083325,None,None,None,None,NaN,NaN,None
5,None,NaN,2.1 DEMOLIÇÃO DE ALVENARIA,4.83975,M³,0.014748,10.0,48.3975,750.0,3629.8125,3678.21,4701.817125,None,None,None,None,NaN,NaN,None
6,None,NaN,2.2 DEMOLIÇÃO DE PISO,8.81000,M²,0.001060,10.0,88.1000,20.0,176.2,264.3,326.851,None,None,None,None,NaN,NaN,None
7,None,NaN,2.3 DEMOLIÇÃO DE ESCADA,1.00000,VB,0.008019,1000.0,1000.0000,1000.0,1000,2000,2430,None,None,None,None,NaN,NaN,None
8,None,NaN,2.4 ABERTURA DE VÃO EM LAJE,1.00000,VB,0.012029,1500.0,1500.0000,1500.0,1500,3000,3645,None,None,None,None,NaN,NaN,None
9,None,NaN,2.5 ABERTURA DE RASGOS DIVERSOS EM ALVENARIA E...,1.00000,VB,0.020048,1000.0,1000.0000,4000.0,4000,5000,6270,None,None,None,None,NaN,NaN,None


## 7. Mapeamento para schema blu

### Sheet `lançamentos` → `fato_transacoes`

| Coluna no arquivo | Campo canonical blu | Observação |
|---|---|---|
| `DATA CUSTO` | `data` | datetime → date |
| `FORNECEDOR` | `fornecedor` | texto livre |
| `NÚMERO   NF` | `numero_nf` | pode ser nulo |
| `VALOR` | `valor` | float |
| `MAT ou MDO` | `tipo` | MAT / MDO |
| `CENTRO DE CUSTO - FAMILIA` | `categoria` | hierarquia nível 1 |
| `CENTRO DE CUSTO - SUB FAMILIA` | `subcategoria` | hierarquia nível 2 |

### Sheet `aditivos` → `fato_transacoes` (tipo = 'ADITIVO')

Mesma estrutura, com campo `DESCRIÇÃO` mapeado para uma nota/observação.

### Sheet `orçamento` → tabela de referência (planejado)

Requer tratamento separado — linhas de subtotal misturadas com linhas de item.

In [13]:
# Construir DataFrame limpo para lançamentos seguindo o mapeamento
df_clean = pd.DataFrame()

df_clean['data'] = pd.to_datetime(df_lanc['DATA CUSTO'], errors='coerce')
df_clean['fornecedor'] = df_lanc['FORNECEDOR'].str.strip()
df_clean['numero_nf'] = df_lanc['NÚMERO   NF']
df_clean['valor'] = pd.to_numeric(df_lanc['VALOR'], errors='coerce')
df_clean['tipo'] = df_lanc['MAT ou MDO'].str.strip().str.upper()
df_clean['categoria'] = df_lanc['CENTRO DE CUSTO - FAMILIA'].str.strip()
df_clean['subcategoria'] = df_lanc['CENTRO DE CUSTO - SUB FAMILIA'].str.strip()
df_clean['fonte'] = 'lancamentos'

df_clean = df_clean.dropna(subset=['data', 'valor'])

print(f'Lançamentos válidos: {len(df_clean)}')
print(f'Período: {df_clean["data"].min().date()} → {df_clean["data"].max().date()}')
print(f'Total R$: R$ {df_clean["valor"].sum():,.2f}')
df_clean.head(10)

Lançamentos válidos: 199
Período: 2025-09-12 → 2026-04-24
Total R$: R$ 100,995.31


,data,fornecedor,numero_nf,valor,tipo,categoria,subcategoria,fonte
0,2025-09-12,BAZAR TODA OBRA,NaN,21.90,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA,lancamentos
1,2025-09-12,A ILUMINADA,NaN,3216.39,MAT,INSTALAÇÕES,INSTALAÇÃO ELÉTRICA,lancamentos
2,2025-09-12,EDSON SANTOS MATERIAL,NaN,30.00,MAT,INSTALAÇÕES,INSTALAÇÃO HIDRÁULICA,lancamentos
3,2025-09-12,EDSON SANTOS MATERIAL,NaN,38.75,MAT,INSTALAÇÕES,INSTALAÇÃO HIDRÁULICA,lancamentos
4,2025-09-12,ÁGUAS CLARA,NaN,36.00,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,OUTROS,lancamentos
5,2025-09-12,ZAMACON,33745.0,240.00,MAT,DEMOLIÇÕES,DEMOLIÇÃO DE ALVENARIA,lancamentos
6,2025-09-12,ZAMACON,33745.0,1058.50,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA,lancamentos
7,2025-09-12,AL GOSTO,NaN,60.00,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,OUTROS,lancamentos
8,2025-09-12,ZAMACON,33745.0,916.80,MAT,MOBILIZAÇÃO E ACOMPANHAMENTO TÉCNICO,"EQUIPAMENTOS, EPIS E EPCS",lancamentos
9,2025-09-15,ZAMACON,33796.0,1049.00,MAT,OBRA CIVIL,ELEVAÇÃO DE ALVENARIA,lancamentos


In [14]:
# Verificar problemas de qualidade
print('=== Qualidade dos dados ===')
print('\nNulos por coluna:')
print(df_clean.isnull().sum())

print(f'\nValores negativos em VALOR: {(df_clean["valor"] < 0).sum()}')
print(f'Fornecedor vazio: {df_clean["fornecedor"].isna().sum() + (df_clean["fornecedor"] == "").sum()}')
print(f'Sem categoria: {df_clean["categoria"].isna().sum()}')

=== Qualidade dos dados ===

Nulos por coluna:
data              0
fornecedor        0
numero_nf       168
valor             0
tipo              0
categoria         0
subcategoria      7
fonte             0
dtype: int64

Valores negativos em VALOR: 0
Fornecedor vazio: 0
Sem categoria: 0


## 8. Problema: detecção automática de header row

O arquivo tem **dois níveis de header** antes dos dados:
- Row 2: labels de seção (`INSUMOS`, `MATERIAL`, `VENDA`) — não são colunas
- Row 3: headers reais

A heurística atual no `upload-csv-source` ("row 0 tem menos colunas que row 1 → skip") **não funciona** para XLSX com múltiplos níveis. Precisamos de uma estratégia diferente.

### Opções:
1. **Deixar o usuário escolher qual row é o header** (mais seguro)
2. **Detectar a row com mais células preenchidas** como candidate header
3. **Para cada sheet, procurar a primeira row onde todos os valores são texto e a próxima tem dados mistos**

In [15]:
# Estratégia: encontrar a row com mais células não-nulas nas primeiras 10 linhas
def detect_header_row(ws, max_search=10):
    """Retorna o índice (0-based) da linha com mais células preenchidas nas primeiras max_search linhas."""
    rows = list(ws.iter_rows(min_row=1, max_row=max_search, values_only=True))
    counts = [(i, sum(1 for v in row if v is not None and str(v).strip() != '')) for i, row in enumerate(rows)]
    # Pegar a que tem mais células preenchidas (ties → primeira)
    best_idx, best_count = max(counts, key=lambda x: x[1])
    print(f'  Células não-nulas por linha: {counts}')
    print(f'  → Header detectado em row {best_idx + 1} ({best_count} células)')
    return best_idx

for name in ['lançamentos', 'aditivos', 'orçamento']:
    print(f"\nSheet '{name}':")
    detect_header_row(wb[name])


Sheet 'lançamentos':
  Células não-nulas por linha: [(0, 0), (1, 3), (2, 9), (3, 7), (4, 7), (5, 7), (6, 7), (7, 7), (8, 8), (9, 8)]
  → Header detectado em row 3 (9 células)

Sheet 'aditivos':
  Células não-nulas por linha: [(0, 0), (1, 2), (2, 10), (3, 0), (4, 0), (5, 0), (6, 0), (7, 0), (8, 0), (9, 0)]
  → Header detectado em row 3 (10 células)

Sheet 'orçamento':
  Células não-nulas por linha: [(0, 0), (1, 1), (2, 17), (3, 6), (4, 7), (5, 10), (6, 10), (7, 10), (8, 7), (9, 10)]
  → Header detectado em row 3 (17 células)


In [16]:
# Confirmar que a estratégia de "max células preenchidas" funciona para lançamentos
ws_lanc = wb['lançamentos']
header_idx = detect_header_row(ws_lanc)

all_rows = list(ws_lanc.iter_rows(values_only=True))
headers = [str(v).strip() if v is not None else f'col_{i}' for i, v in enumerate(all_rows[header_idx])]
print('\nHeaders detectados:', [h for h in headers if not h.startswith('col_')])

  Células não-nulas por linha: [(0, 0), (1, 3), (2, 9), (3, 7), (4, 7), (5, 7), (6, 7), (7, 7), (8, 8), (9, 8)]
  → Header detectado em row 3 (9 células)

Headers detectados: ['MÃO DE OBRA', '#', 'DATA CUSTO', 'FORNECEDOR', 'MAT ou MDO', 'CENTRO DE CUSTO - FAMILIA', 'CENTRO DE CUSTO - SUB FAMILIA', 'NÚMERO   NF', 'VALOR']


## 9. Resumo e recomendações para o pipeline

### Sheets relevantes para ingestão blu:

| Sheet | Uso | Header row | Dados a partir de |
|---|---|---|---|
| `lançamentos` | Custos reais (MAT + MDO) | Row 3 | Row 4 |
| `aditivos` | Materiais extras | Row 3 | Row 4 |
| `orçamento` | Planejado (referência) | Row 3 | Row 5 |
| `resumo` | KPIs consolidados | Row 2 | Row 3 |
| `familias` | Lookup categorias | Row 2 | Row 3 |

### Mudanças necessárias no `upload-csv-source` e frontend:

1. **Para XLSX com múltiplas sheets**: perguntar ao usuário qual sheet usar antes de prosseguir
2. **Detecção de header**: usar "row com mais células preenchidas" nas primeiras 10 linhas (mais robusto que comparar row 0 vs row 1)
3. **Colunas duplas `None`**: filtrar colunas onde o header é `None` ou começa com `col_`
4. **`data_only=True`**: usar ao carregar o XLSX para resolver fórmulas (openpyxl) — o Deno usa a lib `xlsx` que já faz isso por padrão

### Campos que o schema blu precisa para esse tipo de arquivo:

- `tipo_lancamento` (MAT / MDO / ADITIVO) — **não existe no schema atual**
- `categoria` + `subcategoria` (centro de custo hierárquico) — precisa verificar se `dim_inventory` ou campo livre
- `numero_nf` — mapeável para `numero_documento` ou similar